# 실습 2 — Vector Search 2.0 상품 검색 엔진 직접 돌려보기

지금 배포하고 있는 쇼핑 에이전트(`app/embedding_vector.py`)가 런타임에 보내는 것과 **똑같은 API 요청**을, `install.sh`가 미리 만들어 둔 컬렉션 `amazon-product-768-compact`(768차원 dense 필드 2개 + ScaNN 인덱스 2개)에 직접 실행해 봅니다.

> [!IMPORTANT]
> 이 노트북은 컬렉션과 인덱스를 만들지 않습니다. 시작하기 전에 `part2/README.md`의 Cloud Run 배포 명령을 먼저 실행하세요. 배포 빌드(약 5분)는 이 실습을 진행하는 동안 백그라운드에서 함께 돌아갑니다.

## 1. 클라이언트 초기화 및 컬렉션 핸들

앱이 사용하는 것과 동일한 클라이언트 4개를 만듭니다 — 질의 임베딩(`genai`), 벡터 검색과 배치 검색(`DataObjectSearch`), 개별 조회(`DataObject`), 리랭킹(`Rank`).

In [ ]:
import io
import statistics
import urllib.request
from html import escape
from pathlib import Path
from time import perf_counter

import google.auth
from google import genai
from google.genai import types
from google.cloud import vectorsearch_v1beta as vectorsearch
from google.cloud import discoveryengine_v1 as discoveryengine
from IPython.display import HTML, display
from PIL import Image

_, PROJECT_ID = google.auth.default()

# ── app/common.py 와 동일한 값 ──────────────────────────────────────────
LOCATION = "asia-northeast1"
COLLECTION_ID = "amazon-product-768-compact"
COLLECTION_NAME = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}"
IMAGE_SERVER = "https://thumbnail.aidemo.dev"

# ── app/embedding_vector.py 와 동일한 값 ────────────────────────────────
EMBEDDING_MODEL = "gemini-embedding-2"
OUTPUT_DIMENSIONALITY = 768
TEXT_FIELD = "text_embedding"
IMAGE_FIELD = "image_embedding"
SEARCH_TOP_K = 100          # 앱의 SEARCH_TOP_K 와 같은 값입니다.
RANKING_CONFIG = f"projects/{PROJECT_ID}/locations/global/rankingConfigs/default_ranking_config"
TEXT_QUERY_HYBRID_WEIGHTS = [1.35, 0.65]
IMAGE_QUERY_HYBRID_WEIGHTS = [0.65, 1.35]

embedding_client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")
search_client = vectorsearch.DataObjectSearchServiceClient()
data_client = vectorsearch.DataObjectServiceClient()
rank_client = discoveryengine.RankServiceClient()

print("PROJECT_ID :", PROJECT_ID)
print("COLLECTION :", COLLECTION_NAME)

## 2. 백그라운드 인덱싱 완료 확인

`install.sh`는 `session2_index_builder.py`를 백그라운드로 실행합니다. 이 스크립트는 컬렉션 생성, `ImportDataObjects`, ScaNN 인덱스 2개까지 모두 네 개의 장기 실행 작업(LRO)을 만듭니다. 네 작업이 전부 `done` 상태인지 확인합니다.

In [ ]:
!gcloud vector-search operations list --location=asia-northeast1

In [ ]:
# 컬렉션에 실제로 상품이 몇 건 적재되었는지 확인합니다. (벡터 없이 집계만 수행)
try:
    response = search_client.aggregate_data_objects(
        vectorsearch.AggregateDataObjectsRequest(parent=COLLECTION_NAME, aggregate="COUNT")
    )
    # aggregate_results 는 Struct 리스트로 반환되므로 dict 로 변환해야 값을 읽을 수 있습니다.
    rows = [dict(row) for row in response.aggregate_results]
    print("적재된 상품 수 :", rows)
except Exception as exc:  # 임포트가 아직 진행 중이면 여기로 들어옵니다.
    print("❌ 집계 실패:", exc)
    print()
    print("다음 순서로 확인하세요.")
    print("  1) 위 2단계 operations 목록에서 임포트 작업이 done: true 인지 확인")
    print("  2) 터미널에서  tail -30 ~/smx-multimodal-agent/index_builder.log  실행")
    print("  3) 그래도 비어 있으면  bash install.sh  를 다시 실행")


## 3. 컬렉션 스키마 확인 — dense 벡터 필드 2개

`text_embedding`은 상품명과 키워드를 이어 붙인 텍스트를, `image_embedding`은 상품 사진을 임베딩해 둔 필드입니다.
Gemini Embedding 2는 텍스트와 이미지를 같은 벡터 공간에 배치하므로, 질의 벡터 하나로 두 필드를 모두 검색할 수 있습니다.

In [ ]:
try:
    service_client = vectorsearch.VectorSearchServiceClient()
    collection = service_client.get_collection(name=COLLECTION_NAME)
    print("── data_schema (검색 결과와 함께 받을 수 있는 데이터 필드) ──")
    print(collection.data_schema)
    print("── vector_schema (검색 대상 벡터 필드) ──")
    print(collection.vector_schema)
except Exception as exc:
    print("컬렉션 조회 실패:", exc)

## 4. 상품 카탈로그 프리뷰 + 공용 헬퍼 정의

앞으로 계속 사용할 헬퍼를 정의합니다. 각 함수의 docstring에 대응하는 앱 함수를 적어 두었습니다.

**카탈로그 범위** — Amazon Berkeley Objects(ABO) 데이터셋에서 추린 Amazon 자체 브랜드 상품 약 10만 건입니다. 가전·주방·뷰티·식품·가구·신발·가방·주얼리가 중심이며 **의류(원피스·셔츠 등)는 들어 있지 않습니다.** 질의를 바꿔 볼 때는 이 범위 안에서 고르세요. 카탈로그에 없는 카테고리를 물어보면 검색 엔진은 그래도 무언가를 돌려주지만, 점수가 눈에 띄게 낮아집니다.

> 이 컬렉션에는 서버 측 자동 임베딩(`vertex_embedding_config`)이 설정되어 있지 않습니다. 클라이언트가 직접 만든 벡터를 넣고 검색하는 `VectorSearch`(bring-your-own-vector) 방식입니다.

In [ ]:
def embed(text: str | None = None, image: bytes | None = None) -> list[float]:
    """app/embedding_vector.py 의 _embed_with_gemini_embedding_2() 와 동일."""
    contents = text if text is not None else types.Part.from_bytes(data=image, mime_type="image/jpeg")
    response = embedding_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=contents,
        config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMENSIONALITY),
    )
    return list(response.embeddings[0].values)


def to_item(result) -> dict:
    """app/embedding_vector.py 의 _search_result_to_dict() 와 동일."""
    obj = result.data_object
    item_id = obj.data_object_id or obj.name.split("/")[-1]
    return {
        "id": item_id,
        "name": str(obj.data.get("name", "")),
        "description": str(obj.data.get("description", "")),
        "score": result.distance,
    }


def vector_search(embedding, search_field, top_k=SEARCH_TOP_K, metadata_filter=None) -> list[dict]:
    """app/embedding_vector.py 의 _text/_image_similarity_collection_search() 와 동일."""
    clause_kwargs = {
        "search_field": search_field,
        "vector": vectorsearch.DenseVector(values=embedding),
        "top_k": top_k,
        "output_fields": vectorsearch.OutputFields(data_fields=["name", "description"]),
    }
    if metadata_filter is not None:  # 9단계에서 사용합니다.
        clause_kwargs["filter"] = metadata_filter
    request = vectorsearch.SearchDataObjectsRequest(
        parent=COLLECTION_NAME,
        vector_search=vectorsearch.VectorSearch(**clause_kwargs),
    )
    response = search_client.search_data_objects(request)
    return [to_item(result) for result in response.results]


def dedupe(items):
    """같은 상품이 화면을 가득 채우지 않도록 상품명 기준으로 하나만 남깁니다.

    이 카탈로그는 상품 한 개에 사진이 여러 장 있으면 사진마다 별도의 행으로 들어 있고,
    색상·사이즈 변형도 각각 별도의 행입니다. 그래서 검색 상위권은 한두 상품의 사진으로
    가득 찹니다. 검색 자체는 앱과 같은 top_k=100 으로 그대로 두고 화면에 그릴 때만
    걸러냅니다. Part 1에서 만든 비디오 크라우딩 필터와 같은 개념입니다.
    """
    seen, unique = set(), []
    for item in items:
        key = item["name"].strip().lower()
        if key not in seen:
            seen.add(key)
            unique.append(item)
    return unique


def label(name, length):
    """카드에 표시할 상품명을 만듭니다. 앞뒤를 남기고 가운데를 줄입니다.

    이 카탈로그의 상품명은 색상·사이즈가 맨 뒤에 붙습니다. 앞에서부터만 자르면
    변형끼리 완전히 같은 이름으로 보이므로 뒷부분도 함께 남깁니다.
    자르기가 escape() 보다 먼저여야 한다는 점도 중요합니다. 순서가 반대면
    &amp; 나 &#x27; 같은 엔티티가 글자 수를 차지해 이름 뒤쪽이 통째로 사라집니다.
    """
    if len(name) <= length:
        return escape(name)
    head = (length - 1) * 2 // 3
    return escape(name[:head] + "…" + name[head - length + 1:])


def render(items, title="", limit=8):
    """검색 결과를 썸네일 그리드로 표시합니다."""
    items = dedupe(items)
    cards = []
    for rank, item in enumerate(items[:limit], 1):
        cards.append(
            "<div style='width:148px;margin:6px;font-size:11px;text-align:center'>"
            "<img src='{}/{}.webp' style='width:140px;height:140px;object-fit:contain;"
            "background:#fff;border:1px solid #eee'>"
            "<div><b>{}.</b> {}</div><div style='color:#888'>{:.4f}</div></div>".format(
                IMAGE_SERVER, item["id"], rank, label(item["name"], 64), item["score"]
            )
        )
    display(HTML(
        "<b>{}</b><div style='display:flex;flex-wrap:wrap'>{}</div>".format(
            escape(title), "".join(cards))))


PREVIEW_QUERY = "ergonomic office chair with lumbar support"

preview = vector_search(embed(text=PREVIEW_QUERY), TEXT_FIELD)
print("검색 결과 {}건 → 서로 다른 상품 {}종".format(len(preview), len(dedupe(preview))))
render(preview, "카탈로그 프리뷰 — " + PREVIEW_QUERY)

## 5. 텍스트 질의 ➔ `text_embedding` 필드 검색

`find_items` 툴이 받은 영어 쿼리를 처리하는 경로입니다. 두 번째 질의에는 `thermos`·`mug` 같은 단어가 하나도 없지만 보온 텀블러와 진공 보온병이 상위에 올라옵니다.

점수는 이 컬렉션 기준으로 **0.65 이상이면 잘 맞은 것, 0.58 근처면 카탈로그에 마땅한 답이 없다는 신호**입니다. 전체 상품과의 평균 유사도가 0.46 근처라 점수 폭이 좁으니, 절대값보다 상위권과 평균의 간격을 보세요.

In [ ]:
QUERIES = [
    "wireless noise cancelling headphones",
    "something that keeps my coffee hot on the desk all morning",
]

text_results = []
for query in QUERIES:
    embed_started = perf_counter()
    query_vector = embed(text=query)
    embed_ms = (perf_counter() - embed_started) * 1000

    search_started = perf_counter()
    results = vector_search(query_vector, TEXT_FIELD)
    search_ms = (perf_counter() - search_started) * 1000

    print("query={!r}  embed_ms={:.1f}  search_ms={:.1f}  결과 {}건 → 상품 {}종".format(
        query, embed_ms, search_ms, len(results), len(dedupe(results))))
    render(results, "text_embedding ← " + query)
    if not text_results:
        text_results = results

## 6. 이미지 질의 ➔ `image_embedding` 필드 검색 (크로스모달)

같은 이미지 벡터 하나를 ① `image_embedding`(생김새가 닮은 상품) ② `text_embedding`(이미지 벡터로 상품 설명문을 찾는 크로스모달) 두 필드에 각각 보냅니다. 앱의 카메라 경로가 ①입니다.

①은 0.8대, ②는 0.6대의 점수가 나옵니다. 같은 상품군을 찾더라도 이미지끼리 비교할 때 거리가 더 가깝습니다. 두 결과의 구성이 서로 다르다는 점이 다음 단계에서 RRF로 융합하는 이유입니다.

> 질의 이미지는 썸네일 서버에서 내려받은 400px WebP이고 인덱싱에 사용한 것은 원본 사진이므로, 질의로 쓴 상품이 반드시 1위가 되지는 않습니다. 같은 상품의 다른 각도 사진이나 다른 국가 리스팅이 먼저 올라오는 것이 정상입니다.

In [ ]:
def fetch_jpeg(url: str) -> bytes:
    """썸네일을 내려받아 JPEG 바이트로 변환합니다 (앱이 카메라에서 받는 형식과 동일)."""
    with urllib.request.urlopen(url) as response:
        raw = response.read()
    buffer = io.BytesIO()
    Image.open(io.BytesIO(raw)).convert("RGB").save(buffer, format="JPEG")
    return buffer.getvalue()


seed = text_results[0]
seed_url = "{}/{}.webp".format(IMAGE_SERVER, seed["id"])
print("질의 이미지 :", seed["name"])
display(HTML("<img src='{}' width='180' style='border:1px solid #eee'>".format(seed_url)))

image_vector = embed(image=fetch_jpeg(seed_url))
print("이미지 질의 벡터 차원 :", len(image_vector))

render(vector_search(image_vector, IMAGE_FIELD), "① image_embedding ← 이미지 벡터 (생김새)")
render(vector_search(image_vector, TEXT_FIELD), "② text_embedding ← 이미지 벡터 (크로스모달)")

## 7. [핵심] 하이브리드 검색과 RRF 가중치 실험

`batch_search_data_objects`는 검색 절 여러 개를 한 번의 요청으로 실행하고 서버 내장 RRF로 융합합니다.

$$\text{score}(d) = \sum_{i} w_i \cdot \frac{1}{k + \text{rank}_i(d)}$$

Part 1에서 직접 계산한 `alpha`에 해당하는 것이 `weights`이며, **검색 절을 넣은 순서가 곧 가중치 순서**입니다.

| | 1번 절 (`text_embedding`) | 2번 절 (`image_embedding`) |
| :--- | :--- | :--- |
| `TEXT_QUERY_HYBRID_WEIGHTS` | **1.35** | 0.65 |
| `IMAGE_QUERY_HYBRID_WEIGHTS` | 0.65 | **1.35** |

질의 벡터는 그대로 두고 가중치만 뒤집어서, 순위가 어떻게 달라지는지 비교합니다.

In [ ]:
def hybrid_search(embedding, weights, top_k=SEARCH_TOP_K) -> list[dict]:
    """app/embedding_vector.py 의 _hybrid_collection_search() 와 동일한 요청."""
    request = vectorsearch.BatchSearchDataObjectsRequest(
        parent=COLLECTION_NAME,
        searches=[
            vectorsearch.Search(  # 1번 절 → weights[0]
                vector_search=vectorsearch.VectorSearch(
                    search_field=TEXT_FIELD,
                    vector=vectorsearch.DenseVector(values=embedding),
                    top_k=top_k,
                    output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
                )
            ),
            vectorsearch.Search(  # 2번 절 → weights[1]
                vector_search=vectorsearch.VectorSearch(
                    search_field=IMAGE_FIELD,
                    vector=vectorsearch.DenseVector(values=embedding),
                    top_k=top_k,
                    output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
                )
            ),
        ],
        combine=vectorsearch.BatchSearchDataObjectsRequest.CombineResultsOptions(
            ranker=vectorsearch.Ranker(
                rrf=vectorsearch.ReciprocalRankFusion(weights=weights)
            ),
            output_fields=vectorsearch.OutputFields(data_fields=["name", "description"]),
            top_k=top_k,
        ),
    )
    response = search_client.batch_search_data_objects(request)
    fused = response.results[0].results if response.results else []
    return [to_item(result) for result in fused]


def compare(left_title, left, right_title, right, limit=8):
    """두 결과 목록을 나란히 놓고 순위 변동을 표시합니다."""
    left, right = dedupe(left), dedupe(right)
    left_rank = {item["id"]: i for i, item in enumerate(left, 1)}
    right_rank = {item["id"]: i for i, item in enumerate(right, 1)}

    def badge(item, rank, other):
        previous = other.get(item["id"])
        if previous is None:
            return "<span style='color:#c0392b'>NEW</span>"
        if previous == rank:
            return "<span style='color:#aaa'>=</span>"
        if previous > rank:
            return "<span style='color:#1e8449'>▲{}</span>".format(previous - rank)
        return "<span style='color:#2471a3'>▼{}</span>".format(rank - previous)

    def cells(items, rank, other):
        if rank > len(items):
            return "<td></td><td></td>"
        item = items[rank - 1]
        return ("<td style='padding:4px'><img src='{}/{}.webp' width='52' "
                "style='object-fit:contain;background:#fff'></td>"
                "<td style='padding:4px;font-size:11px'>{} {}</td>").format(
                    IMAGE_SERVER, item["id"], label(item["name"], 46), badge(item, rank, other))

    rows = []
    for rank in range(1, limit + 1):
        rows.append("<tr><td style='padding:4px;color:#888'>{}</td>{}{}</tr>".format(
            rank, cells(left, rank, right_rank), cells(right, rank, left_rank)))
    display(HTML(
        "<table style='border-collapse:collapse'>"
        "<tr><th></th><th colspan='2' style='padding:6px'>{}</th>"
        "<th colspan='2' style='padding:6px'>{}</th></tr>{}</table>".format(
            escape(left_title), escape(right_title), "".join(rows))))


text_weighted = hybrid_search(image_vector, TEXT_QUERY_HYBRID_WEIGHTS)
image_weighted = hybrid_search(image_vector, IMAGE_QUERY_HYBRID_WEIGHTS)

compare(
    "TEXT_QUERY_HYBRID_WEIGHTS = [1.35, 0.65]", text_weighted,
    "IMAGE_QUERY_HYBRID_WEIGHTS = [0.65, 1.35]", image_weighted,
)
print("▲▼ 는 반대편 목록과 비교한 순위 변동, NEW 는 반대편 상위 8위 안에 없던 상품입니다.")

> 가중치 한쪽을 0으로 주면(`[2.0, 0.0]`, `[0.0, 2.0]`) 그 절은 결과에 아무 영향도 주지 못하고 단일 필드 검색과 같아집니다. 어느 한쪽 극단도 정답이 아니기 때문에 하이브리드를 사용합니다.

## 8. Ranking API 리랭킹

벡터 검색은 재현율(recall)을, Ranking API는 정밀도(precision)를 맡습니다. 벡터 검색으로 후보 100건을 근사로 확보한 다음, Ranking API가 질의와 상품 설명문을 함께 읽는 교차 인코더(cross-encoder)로 다시 점수를 매겨 정렬합니다. `find_items` 툴이 `ranking_query`를 따로 받는 이유입니다.

In [ ]:
def rank_results(query: str, results: list[dict]) -> list[dict]:
    """app/embedding_vector.py 의 _rank_results() 와 동일 (원본은 리스트를 제자리 정렬)."""
    if not results or not query:
        return results
    records = [
        discoveryengine.RankingRecord(
            id=item["id"], title=item["name"], content=item.get("description", "")
        )
        for item in results
    ]
    response = rank_client.rank(
        request=discoveryengine.RankRequest(
            ranking_config=RANKING_CONFIG,
            query=query,
            records=records,
            top_n=len(records),
        )
    )
    scores = {record.id: record.score for record in response.records}
    ranked = [dict(item, score=scores.get(item["id"], 0.0)) for item in results]
    ranked.sort(key=lambda item: item["score"], reverse=True)
    return ranked


# 6단계의 질의 이미지는 QUERIES[0] 의 검색 결과에서 골랐습니다.
# 리랭킹 질의도 의도가 같아야 순위 변동이 의미를 가집니다.
RANKING_QUERY = QUERIES[0]

reranked = rank_results(RANKING_QUERY, text_weighted)
compare("RRF 융합 직후", text_weighted, "Ranking API 리랭킹 후 — " + RANKING_QUERY, reranked)
print("점수 기준도 함께 바뀝니다: RRF 융합 점수 → Ranking API 관련도 점수(0~1).")

## 9. 메타데이터 필터 결합

VS2 필터는 MongoDB 스타일 JSON이며 검색 절마다 따로 지정합니다 (`$eq`, `$ne`, `$lt`, `$gt`, `$in`, `$and`, `$or`).

```python
filter={"$and": [{"category": {"$eq": "Shorts"}}, {"retail_price": {"$lt": 30}}]}
```

> 이 컬렉션의 `data_schema`에는 `name`, `description` 두 개뿐이라 예제도 `name`으로 거릅니다. 실제 서비스라면 필터에 쓸 속성을 `data_schema`에 넣고, 인덱스를 만들 때 `filter_fields=[...]`로 선언해야 대규모에서도 빠릅니다.

In [ ]:
allowed_names = [item["name"] for item in dedupe(preview)[:3]]
print("필터로 허용할 상품 3종:")
for name in allowed_names:
    print("  -", name[:70])

filtered = vector_search(
    embed(text=PREVIEW_QUERY),
    TEXT_FIELD,
    metadata_filter={"name": {"$in": allowed_names}},
)
print("\n필터 적용 후 결과 {}건 → 상품 {}종".format(len(filtered), len(dedupe(filtered))))
print("유사도가 아무리 높아도 허용 목록 밖 상품은 후보에 아예 들어오지 않습니다.")
render(filtered, "text_embedding + filter={'name': {'$in': [...]}}")

## 10. ANN(ScaNN) vs kNN — 코드는 그대로, 속도만 바뀐다

| | Part 1 미디어 컬렉션 | Part 2 상품 컬렉션 |
| :--- | :--- | :--- |
| 인덱스 | 없음 | ScaNN 2개 |
| 검색 방식 | kNN 완전탐색 | ANN 근사탐색 |
| 데이터 규모 | 약 1,000건 | 약 10만 건 |
| 질의 코드 | `SearchDataObjectsRequest(...)` | **완전히 동일** |

인덱스는 별도의 검색 엔드포인트가 아닙니다. 검색 요청은 언제나 컬렉션으로 보내고, 질의한 필드에 인덱스가 있으면 서버가 알아서 사용합니다.

In [ ]:
def find_file(*candidates) -> Path:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(candidates)


def show_function(path: Path, name: str) -> None:
    lines = path.read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if line.startswith("def " + name + "("))
    end = start + 1
    while end < len(lines):
        line = lines[end]
        if line.strip() and not line[:1].isspace():   # 들여쓰기가 끝나면 함수도 끝
            break
        end += 1
    print("─" * 78)
    print("{}  ::  {}()".format(path, name))
    print("─" * 78)
    print("\n".join(lines[start:end]).rstrip())


BUILDER_PY = find_file("../session2_index_builder.py", "session2_index_builder.py")
show_function(BUILDER_PY, "request_index")

# 이 컬렉션에 실제로 어떤 인덱스가 붙어 있는지 먼저 확인합니다.
# 인덱스가 없는 필드로 검색하면 kNN 완전탐색으로 처리되므로 아래 지연시간의 의미가 달라집니다.
indexes = list(service_client.list_indexes(parent=COLLECTION_NAME))
if indexes:
    for index in indexes:
        print("인덱스 {}  ← index_field={}  store_fields={}".format(
            index.name.split("/")[-1], index.index_field, list(index.store_fields)))
else:
    print("⚠️ 인덱스가 아직 없습니다. 아래 지연시간은 ANN이 아니라 kNN 완전탐색 수치입니다.")
print()

# 질의의 실제 지연시간을 측정합니다.
probe_vector = embed(text="gold stud earrings")
latencies = []
for _ in range(5):
    started = perf_counter()
    vector_search(probe_vector, TEXT_FIELD)
    latencies.append((perf_counter() - started) * 1000)

print("\nsearch_ms 5회 :", ", ".join("{:.1f}".format(value) for value in latencies))
print("중앙값        : {:.1f} ms  (임베딩 생성 시간 제외, 순수 검색)".format(statistics.median(latencies)))

## 11. `app/embedding_vector.py` 소스 대조

노트북의 `hybrid_search()`와 앱의 `_hybrid_collection_search()`를 나란히 출력합니다. 검색 절 순서, `weights`, 결과 파싱까지 같은 요청이고 로깅 여부만 다릅니다. 다만 이 앱 함수는 런타임에 호출되지 않습니다.

In [ ]:
import inspect

APP_EMBEDDING_PY = find_file(
    "app/embedding_vector.py",
    "../part2/app/embedding_vector.py",
)

show_function(APP_EMBEDDING_PY, "_hybrid_collection_search")
print()
print("─" * 78)
print("이 노트북의 hybrid_search()")
print("─" * 78)
print(inspect.getsource(hybrid_search).rstrip())

> [!IMPORTANT]
> ### 재현율 vs 지연 — 이 앱이 실제로 내린 선택
>
> `_hybrid_collection_search()`는 구현되어 있지만 실행되지 않습니다. `_collection_search()`가 텍스트 단독 경로에서 바로 반환하고 하이브리드 호출부는 주석 처리되어 있습니다 — 커밋 `4bc2657 "Set use only text for latency reduce"`에서 의도적으로 바꾼 것입니다.
>
> - **잃은 것**: 크로스모달 재현율. `image_embedding` 쪽 후보를 통째로 보지 못합니다.
> - **얻은 것**: 서버 측 벡터 검색이 2회에서 1회로 줄었습니다. Live 음성 대화에서는 수백 ms가 대화의 자연스러움을 좌우하고, `find_items`는 쿼리를 여러 개 동시에 보냅니다.
> - **무관**: 카메라 경로는 원래부터 `image_embedding` 단독입니다.
>
> 검색 설계는 기능을 전부 켜는 일이 아니라, 재현율을 얼마나 포기하고 지연시간을 얼마나 줄일지 고르는 일입니다.

In [ ]:
show_function(APP_EMBEDDING_PY, "_collection_search")

## 12. 에이전트 프롬프트와 `find_items` 툴 호출 흐름

```
경로 A — 카메라 프레임 (자동)
  JPEG → 유사상품 워커 스레드 → _image_similarity_search()
       → image_embedding 단독 검색 [6단계 ①] → 좌측 타일 실시간 갱신

경로 B — 음성 발화 → 툴 호출
  find_items(queries=[영어 쿼리 N개], ranking_query="영어 요약")
    → 쿼리별 스레드 병렬 _collection_search(text=q)   [5단계]
    → id 중복 제거 → _rank_results()                  [8단계]
    → 상위 64건 렌더링 + 음성 브리핑
```

프롬프트는 검색 쿼리를 **영어**로 만들게 하고(카탈로그 텍스트가 영어 중심), 음성 응답은 **한국어**로만 하도록 강하게 지시합니다.

> 앱은 `id`가 같은 것만 중복으로 걸러냅니다. 그래서 같은 상품의 다른 사진이 타일에 여러 번 나올 수 있습니다. 4단계에서 정의한 `dedupe()`는 이 노트북에서만 하는 처리입니다.

In [ ]:
APP_PROMPT_PY = find_file(
    "app/prompt.py",
    "../part2/app/prompt.py",
)

prompt_source = APP_PROMPT_PY.read_text()
step1 = "## 1단계" + prompt_source.split("## 1단계", 1)[1].split("## 2단계", 1)[0]
print("─" * 78)
print("{}  ::  AGENT_PROMPT 발췌".format(APP_PROMPT_PY))
print("─" * 78)
print(step1.rstrip())

## 실습 2 완료 🎉

1. 질의 코드는 Part 1과 같고, 달라진 것은 컬렉션에 붙은 ScaNN 인덱스뿐이다.
2. 벡터 하나를 `text_embedding` / `image_embedding` 두 필드에 모두 보낼 수 있다.
3. RRF `weights`가 Part 1의 `alpha`를 대신하며, 가중치를 뒤집으면 순위가 바뀐다.
4. Ranking API가 재현율 위주로 모은 결과를 정밀도 위주로 다시 세운다.
5. 실제 서비스는 이 손잡이를 전부 켜지 않는다 — 검색 설계는 트레이드오프를 고르는 일이다.

`part2/README.md`로 돌아가 배포 상태를 확인하고, QR 코드로 에이전트를 직접 사용해 봅니다.